# Qwen 2.5 0.5B RL & Tool Fine-Tuning Pipeline (Google Colab T4)

This notebook trains an autonomous **Tool-Augmented Traffic SOP Dispatcher Policy** based on `Qwen/Qwen2.5-0.5B-Instruct` using **SFT + QLoRA** on a free Google Colab T4 GPU, and pushes the fine-tuned checkpoint directly to **Hugging Face Model Hub (`HamzaBoy/qwen2.5-0.5b-traffic-sop`)**.

In [ ]:
# 1. Install required dependencies on Colab T4
!pip install -q trl peft transformers datasets bitsandbytes networkx huggingface_hub torch

In [ ]:
# 2. Authenticate with Hugging Face Hub
import os
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Enter your Hugging Face WRITE Token: ")
login(token=HF_TOKEN)
print("✅ Logged in to Hugging Face Hub!")

In [ ]:
# 3. Load SFT Dataset directly from Hugging Face Hub
import json
from huggingface_hub import hf_hub_download
from datasets import Dataset

HUB_REPO_ID = "HamzaBoy/qwen2.5-0.5b-traffic-sop"
print(f"Downloading sft_traffic_sop_train.jsonl directly from {HUB_REPO_ID}...")
dataset_file_path = hf_hub_download(repo_id=HUB_REPO_ID, filename="sft_traffic_sop_train.jsonl", repo_type="model", token=HF_TOKEN)

records = []
with open(dataset_file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"✅ Successfully loaded {len(records)} valid SFT tool-calling trajectories!")
dataset = Dataset.from_list(records)
print("Sample Record User Prompt:", dataset[0]["messages"][1]["content"])

In [ ]:
# 4. Initialize Base Model & LoRA Config (4-bit QLoRA mode for Colab T4 GPU)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
print("Base Model & PeftConfig initialized!")

In [ ]:
# 5. Stage 1: Supervised Fine-Tuning (SFT) - Disables GradScaler AMP for T4 Compatibility
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./qwen2.5-0.5b-traffic-sop-sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    peft_config=peft_config,
)

print("=== Starting Stage 1 SFT Training ===")
trainer.train()
print("✅ SFT Training Complete!")

In [ ]:
# 6. Push Fine-Tuned Model Adapter to Hugging Face Hub
HUB_REPO_ID = "HamzaBoy/qwen2.5-0.5b-traffic-sop"
print(f"Pushing fine-tuned model adapter to {HUB_REPO_ID}...")
trainer.model.push_to_hub(HUB_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_REPO_ID, token=HF_TOKEN)
print(f"🎉 Model successfully uploaded to Hugging Face Hub: https://huggingface.co/{HUB_REPO_ID}")